# Using the RefQA dataset

**RefQA** is a million-scale cross-document clinical citation–evidence Q&A dataset (1,524,426 records) built from PubMed Central. This notebook shows how to **download**, **load**, and **use** the released data.

- **Dataset (Zenodo):** https://doi.org/10.5281/zenodo.20805692
- **Code (this repo):** the data-construction pipeline (to *reproduce* the corpus from raw PMC; most users do not need this).
- **Full documentation:** the companion *Scientific Data* Data Descriptor.

> Requirements for this notebook: `pip install datasets pyarrow requests`

## 1. Download from Zenodo

The deposit contains **four files**:

| File | Contents |
|---|---|
| `full_qa_clinical_v03_unique.jsonl` | Full canonical corpus, 1,524,426 records, one JSON object per line (42 fields). |
| `v03_unique_split_index.tsv` | Deterministic train/validation/test/gold/fail split index, keyed by `record_key`. |
| `refqa_parquet_v1.0.tar.gz` | Same corpus as Parquet; extracts to `parquet/permissive/` (561,795 records) and `parquet/full/` (1,524,426). |
| `refqa_aux_v1.0.tar.gz` | Validation bundle: `gold/` (gold 1,000; annotated 250; adversarial 10; frozen overlap), `prompts/` (frozen system prompt), `runtime/` (Technical Validation outputs). |

In [ ]:
import requests, pathlib, shutil

RECORD = "20805692"  # Zenodo record id (DOI 10.5281/zenodo.20805692)
dest = pathlib.Path("refqa_data"); dest.mkdir(exist_ok=True)

meta = requests.get(f"https://zenodo.org/api/records/{RECORD}").json()
for f in meta["files"]:
    name, url = f["key"], f["links"]["self"]
    out = dest / name
    if out.exists():
        print("exists:", name); continue
    print("downloading:", name)
    with requests.get(url, stream=True) as r, open(out, "wb") as fh:
        shutil.copyfileobj(r.raw, fh)
print("done ->", dest.resolve())

In [ ]:
import tarfile
# extract the two archives (Parquet shards + validation bundle)
for arc in ["refqa_parquet_v1.0.tar.gz", "refqa_aux_v1.0.tar.gz"]:
    p = dest / arc
    if p.exists():
        with tarfile.open(p) as t: t.extractall(dest)
        print("extracted:", arc)
# now available: refqa_data/parquet/{permissive,full}/ , refqa_data/gold/ , refqa_data/prompts/ , refqa_data/runtime/

## 2. Load the data

The **permissive** configuration (CC-BY family + public domain) is the recommended default. Use the **full** corpus together with the `license_bucket` column to filter for your own redistribution needs. The gold holdout is a separate shard and is excluded from the train/validation/test splits below.

In [3]:
from datasets import load_dataset

# Recommended: permissive subset, as train/validation/test
ds = load_dataset("parquet", data_files={
    "train":      "refqa_data/parquet/permissive/train-*.parquet",
    "validation": "refqa_data/parquet/permissive/val-*.parquet",
    "test":       "refqa_data/parquet/permissive/test-*.parquet",
})
print(ds)
print(ds["train"][0])

# Full corpus (all 1,524,426 records); filter by license_bucket as needed:
# ds_full = load_dataset("parquet", data_files={"train": "refqa_data/parquet/full/train-*.parquet", ...})
# The JSONL is equivalent: [json.loads(l) for l in open('refqa_data/full_qa_clinical_v03_unique.jsonl')]

DatasetDict({
    train: Dataset({
        features: ['record_key', 'citing_pmcid', 'citing_pmid', 'citing_doi', 'citing_journal', 'citing_year', 'citing_subset', 'section', 'section_bucket', 'paragraph_idx', 'evidence_sentence', 'n_co_cited_refs', 'cited_ref_id', 'cited_pmid', 'cited_doi', 'cited_title', 'cited_abstract', 'cited_journal', 'cited_year', 'cited_mesh_json', 'cited_pub_types_json', 'icite_citation_count', 'icite_relative_citation_ratio', 'icite_nih_percentile', 'icite_is_clinical', 'icite_is_research_article', 'relevance_check', 'mismatch_reason', 'attributed_claim', 'citation_purpose', 'evidence_type', 'support_strength', 'cited_study_design', 'question', 'answer', 'answer_grounded_in_json', 'evidence_quote_from_cited', 'statistic_extraction_json', 'license_raw', 'license_bucket', 'is_permissive', 'split'],
        num_rows: 477999
    })
    validation: Dataset({
        features: ['record_key', 'citing_pmcid', 'citing_pmid', 'citing_doi', 'citing_journal', 'citing_year

## 3. What you can do with RefQA

A quick tour of the main use cases (see the Data Descriptor *Usage Notes* for detail).

### 3a. Citation-grounded clinical question answering
Each supported record carries a `question`, an `answer` (with an inline `[PMID:...]` pointer), and a verbatim `evidence_quote_from_cited`. Train or benchmark models to answer using the cited evidence.

In [4]:
rec = next(r for r in ds["train"] if r["relevance_check"] == "match" and r["question"] and r["evidence_quote_from_cited"])
for k in ["question", "answer", "evidence_quote_from_cited", "citation_purpose", "evidence_type", "support_strength"]:
    print(f"{k}: {rec[k]}")

question: What methodology was used to perform the mosquito membrane feeds in this study?
answer: The mosquito membrane feeds were performed using a protocol previously described elsewhere, specifically referencing the experimental infection methods used with Anopheles gambiae mosquitoes in a prior randomized controlled trial [PMID:14728607].
evidence_quote_from_cited: Experimental infections of Anopheles gambiae s.s. mosquitoes were performed with blood from patients
citation_purpose: method
evidence_type: method-citation
support_strength: moderate


### 3b. Claim–evidence faithfulness
When the cited paper supports the claim, `evidence_quote_from_cited` is a **verbatim substring** of `cited_abstract`, so faithfulness is checkable by a substring match (loose = NFKC + lowercase + whitespace/hyphen normalisation).

In [5]:
import re, unicodedata
def norm(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    s = re.sub(r"[\u2010-\u2015-]", "-", s)
    return re.sub(r"\s+", " ", s).strip()

q = rec.get("evidence_quote_from_cited")
if q:
    print("quote grounded in abstract:", norm(q) in norm(rec["cited_abstract"]))

quote grounded in abstract: True


### 3c. Hallucination / mismatch detection
Records with `relevance_check == "mismatch"` are a natural negative class (the cited paper does *not* support the citing claim). The `gold/RefQA-Adversarial.jsonl` set (10 records) contains mismatches confirmed by **both** human reviewers — a small, reliable held-out adversarial set for retrieval-augmented-generation evaluation.

In [6]:
import json
adv = [json.loads(l) for l in open("refqa_data/gold/RefQA-Adversarial.jsonl")]
print(len(adv), "gold-confirmed adversarial mismatches")
print(adv[0]["mismatch_reason"])

10 gold-confirmed adversarial mismatches
The citing context references an adaptive dose-escalation trial design, but the cited paper is a multicenter phase II study of sequential gemcitabine and docetaxel chemotherapy in elderly NSCLC patients, which contains no mention of adaptive dose-escalation methodology.


### 3d. Gold subset & reproducible splits
`gold/gold_validation_annotated.jsonl` (250 records) carries a human-consolidated `ground_truth`. The `gold` split is held out from training; keep it separate for any quality-comparison. Splits are reproducible from `v03_unique_split_index.tsv` (join on `record_key`).

### 3e. Supervised fine-tuning (chat format)
A RefQA record maps naturally to a chat example: the frozen system prompt (`prompts/v03_system_prompt.txt`) is the system turn, the enriched record is the user turn, and the CitationQA JSON is the assistant turn.

In [7]:
system = open("refqa_data/prompts/v03_system_prompt.txt").read()
import json
SCALAR = ["relevance_check","mismatch_reason","attributed_claim","citation_purpose",
          "evidence_type","support_strength","cited_study_design","question","answer",
          "evidence_quote_from_cited"]
def to_chat(rec):
    # the Parquet columns store the two non-scalar fields as JSON strings
    assistant = {k: rec[k] for k in SCALAR}
    assistant["answer_grounded_in"] = json.loads(rec.get("answer_grounded_in_json") or "[]")
    assistant["statistic_extraction"] = (json.loads(rec["statistic_extraction_json"])
                                          if rec.get("statistic_extraction_json") else None)
    user = {"citing_sentence": rec["evidence_sentence"], "cited_abstract": rec["cited_abstract"]}
    return [{"role": "system", "content": system},
            {"role": "user", "content": json.dumps(user, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(assistant, ensure_ascii=False)}]

chat = to_chat(rec)
print([m["role"] for m in chat])
print("assistant turn (truncated):", chat[2]["content"][:220])

['system', 'user', 'assistant']
assistant turn (truncated): {"relevance_check": "match", "mismatch_reason": "", "attributed_claim": "The membrane feeding procedure and mosquito dissection method used in this study followed a protocol previously described in the cited work.", "cit


---
To **reproduce** the corpus from raw PubMed Central instead of using the release, see the pipeline scripts in this repository (`pipeline/`) and the Data Descriptor Methods.